# Module 084: ORMs: Introduction to SQLAlchemy

Phase 9: Databases & Web Apps | Duration: 2.5 hours

## Setup: Engine and Declarative Base

In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey, DateTime
from sqlalchemy.orm import declarative_base, relationship, sessionmaker
from datetime import datetime

engine = create_engine('sqlite:///:memory:', echo=False)
Base = declarative_base()

## Defining Models

In [ ]:
class Author(Base):
    __tablename__ = 'authors'
    id = Column(Integer, primary_key=True)
    name = Column(String(100), nullable=False)
    birth_year = Column(Integer)
    books = relationship('Book', back_populates='author')

class Book(Base):
    __tablename__ = 'books'
    id = Column(Integer, primary_key=True)
    title = Column(String(200), nullable=False)
    year = Column(Integer)
    price = Column(Float, default=0.0)
    author_id = Column(Integer, ForeignKey('authors.id'))
    author = relationship('Author', back_populates='books')

Base.metadata.create_all(engine)
print('Models created and tables generated')

## Creating a Session and Inserting Data

In [ ]:
Session = sessionmaker(bind=engine)
session = Session()

author: Author = Author(name='J.K. Rowling', birth_year=1965)
session.add(author)
session.commit()
print(f'Author created with id: {author.id}')

book1: Book = Book(title='Harry Potter and the Sorcerer\'s Stone', year=1997, price=24.99, author=author)
book2: Book = Book(title='Harry Potter and the Chamber of Secrets', year=1998, price=24.99, author=author)
session.add_all([book1, book2])
session.commit()
print(f'Books created: {book1.id}, {book2.id}')

## Querying with filter and order_by

In [ ]:
books: list[Book] = session.query(Book).filter(Book.price > 20).order_by(Book.year).all()
for book in books:
    print(f'{book.title} ({book.year}) - ${book.price:.2f}')

## Using Relationships

In [ ]:
# Access author from book
book: Book = session.query(Book).first()
print(f'Book: {book.title}, Author: {book.author.name}')

# Access books from author
author: Author = session.query(Author).first()
print(f'Author: {author.name}, Books:')
for b in author.books:
    print(f'  - {b.title}')

## Updating and Deleting

In [ ]:
# Update
book.price = 29.99
session.commit()
print(f'Updated price: ${book.price:.2f}')

# Delete
session.delete(book2)
session.commit()
remaining: int = session.query(Book).count()
print(f'Remaining books: {remaining}')

session.close()